In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
import joblib
import os

# Pre-define for Pylance
TIER_ML   = []
TIER_DOW  = []
TIER_MED  = []
TIER_MEAN = []
TIER_ZERO = []
FEAT_COLS = []
TOP30     = []

grid     = pd.read_parquet('../data/processed/daily_sales.parquet')
sku_meta = pd.read_parquet('../data/processed/sku_metadata.parquet')

print(f'Grid shape : {grid.shape}')
print(f'SKU meta   : {sku_meta.shape}')
print()
print('Strategy distribution:')
print(sku_meta['strategy'].value_counts())

Grid shape : (1754, 15972)
SKU meta   : (15972, 12)

Strategy distribution:
strategy
zero        15920
median         33
ML_model       19
Name: count, dtype: int64


In [3]:
# ML model → AX, AY, AZ (high business value regardless of behavior)
TIER_ML   = sku_meta[sku_meta['strategy'] == 'ML_model'].index.tolist()

# DOW average → BX, BY
TIER_DOW  = sku_meta[sku_meta['strategy'] == 'DOW_avg'].index.tolist()

# Median → BZ (intermittent medium value)
TIER_MED  = sku_meta[sku_meta['strategy'] == 'median'].index.tolist()

# Simple mean → CX, CY
TIER_MEAN = sku_meta[sku_meta['strategy'] == 'mean'].index.tolist()

# Zero → CZ
TIER_ZERO = sku_meta[sku_meta['strategy'] == 'zero'].index.tolist()

# Top 30 by revenue for oversampling
TOP30 = (sku_meta['total_revenue']
         .sort_values(ascending=False)
         .head(30).index.tolist())

print(f'ML model   : {len(TIER_ML):>5} SKUs  (AX, AY, AZ)')
print(f'DOW avg    : {len(TIER_DOW):>5} SKUs  (BX, BY)')
print(f'Median     : {len(TIER_MED):>5} SKUs  (BZ)')
print(f'Mean       : {len(TIER_MEAN):>5} SKUs  (CX, CY)')
print(f'Zero       : {len(TIER_ZERO):>5} SKUs  (CZ)')
print(f'Top 30     : {len(TOP30):>5} SKUs  ← oversampled 3x')

ML model   :    19 SKUs  (AX, AY, AZ)
DOW avg    :     0 SKUs  (BX, BY)
Median     :    33 SKUs  (BZ)
Mean       :     0 SKUs  (CX, CY)
Zero       : 15920 SKUs  (CZ)
Top 30     :    30 SKUs  ← oversampled 3x


In [4]:
LAG_DAYS     = [1, 2, 3, 7, 14, 21, 28, 35, 42, 56, 364, 365, 366]
ROLL_WINDOWS = [7, 14, 28, 56, 90]
DOW_LAG_WKS  = [1, 2, 4, 8, 12]

def build_features(d, series, meta_row):
    hist = series[series.index < d]

    row = {
        # ── 1. TEMPORAL ────────────────────────────────────────
        'dow'            : d.dayofweek,
        'month'          : d.month,
        'dom'            : d.day,
        'quarter'        : d.quarter,
        'week'           : int(d.isocalendar().week),
        'is_weekend'     : int(d.dayofweek >= 5),
        'is_tet'         : int((d.month==1 and d.day>=15) or
                               (d.month==2 and d.day<=15)),
        'is_month_end'   : int(d.day >= 25),
        'is_month_start' : int(d.day <= 5),
        'is_q1'          : int(d.month in [1, 2, 3]),
        'is_q2'          : int(d.month in [4, 5, 6]),
        'is_q3'          : int(d.month in [7, 8, 9]),
        'is_q4'          : int(d.month in [10, 11, 12]),
        'is_end_of_q'    : int(d.month in [3, 6, 9, 12]
                               and d.day >= 20),
        'is_start_of_q'  : int(d.month in [1, 4, 7, 10]
                               and d.day <= 10),

        # ── 2. BUSINESS FEATURES ───────────────────────────────
        'rev_share'      : float(meta_row['rev_share']),
        'total_revenue'  : float(meta_row['total_revenue']),
        'abc_A'          : int(meta_row['abc'] == 'A'),
        'abc_B'          : int(meta_row['abc'] == 'B'),
        'abc_C'          : int(meta_row['abc'] == 'C'),

        # ── 3. DEMAND BEHAVIOR ─────────────────────────────────
        'cv'             : float(meta_row['cv']),
        'zero_ratio'     : float(meta_row['zero_ratio']),
        'adi'            : float(meta_row['adi'])
                           if not np.isnan(meta_row['adi']) else 999.,
        'days_active'    : float(meta_row['days_active']),
        'xyz_X'          : int(meta_row['xyz'] == 'X'),
        'xyz_Y'          : int(meta_row['xyz'] == 'Y'),
        'xyz_Z'          : int(meta_row['xyz'] == 'Z'),

        # ── 4. SPARSITY FEATURES ───────────────────────────────
        'zero_frac_7'    : float((hist.iloc[-7:] == 0).mean())
                           if len(hist) >= 7  else 1.,
        'zero_frac_28'   : float((hist.iloc[-28:] == 0).mean())
                           if len(hist) >= 28 else 1.,
        'zero_frac_90'   : float((hist.iloc[-90:] == 0).mean())
                           if len(hist) >= 90 else 1.,
    }

    # Days since last sale
    nonzero_idx = hist[hist > 0].index
    row['days_since_last_sale'] = float(
        (d - nonzero_idx[-1]).days
    ) if len(nonzero_idx) > 0 else 999.

    # Active ratio (last 90 days)
    row['active_ratio_90'] = float(
        (hist.iloc[-90:] > 0).mean()
    ) if len(hist) >= 90 else 0.

    # ── 5. POINT LAGS ──────────────────────────────────────────
    for lag in LAG_DAYS:
        row[f'lag_{lag}'] = float(
            series.get(d - pd.Timedelta(days=lag), 0.0))

    # ── 6. ROLLING STATS ───────────────────────────────────────
    for w in ROLL_WINDOWS:
        win = hist.iloc[-w:] if len(hist) >= w else hist
        row[f'rmean_{w}'] = float(win.mean())
        row[f'rstd_{w}']  = float(win.std())  if len(win) > 1 else 0.
        row[f'rmax_{w}']  = float(win.max())
        row[f'rpos_{w}']  = float((win > 0).mean())

    # ── 7. QUARTERLY SEASONALITY ───────────────────────────────
    same_q = hist[hist.index.quarter == d.quarter]
    row['q_mean']    = float(same_q.mean())    if len(same_q) > 0 else 0.
    row['q_std']     = float(same_q.std())     if len(same_q) > 1 else 0.
    row['q_max']     = float(same_q.max())     if len(same_q) > 0 else 0.

    last_yr_q = hist[
        (hist.index.quarter == d.quarter) &
        (hist.index.year    == d.year - 1)
    ]
    row['q_ly_mean'] = float(last_yr_q.mean()) if len(last_yr_q) > 0 else 0.
    row['q_ly_sum']  = float(last_yr_q.sum())  if len(last_yr_q) > 0 else 0.

    # ── 8. SAME WEEKDAY LAGS ───────────────────────────────────
    same_dow = hist[hist.index.dayofweek == d.dayofweek]
    for wk in DOW_LAG_WKS:
        row[f'dlag_{wk}w'] = float(same_dow.iloc[-wk]) \
                              if len(same_dow) >= wk else 0.

    # ── 9. TREND ───────────────────────────────────────────────
    for tw, key in [(28, 'trend_28'), (90, 'trend_90')]:
        seg = hist.iloc[-tw:]
        row[key] = float(
            np.polyfit(np.arange(len(seg)),
                       seg.values.astype(float), 1)[0]
        ) if len(seg) > 2 else 0.

    # ── 10. SAME PERIOD LAST YEAR ──────────────────────────────
    ly     = d - pd.DateOffset(years=1)
    ly_win = hist[
        (hist.index >= ly - pd.Timedelta(days=14)) &
        (hist.index <= ly + pd.Timedelta(days=14))
    ]
    row['ly_mean'] = float(ly_win.mean()) if len(ly_win) > 0 else 0.

    return row

# Test
_test = build_features(
    pd.Timestamp('2025-08-01'),
    grid[grid.columns[0]],
    sku_meta.iloc[0]
)
print(f'build_features ready — {len(_test)} features')

build_features ready — 78 features


In [5]:
X_rows, y_rows = [], []

print(f'Building training data for {len(TIER_ML)} ML SKUs...')

for i, sku in enumerate(TIER_ML):
    series   = grid[sku]
    meta_row = sku_meta.loc[sku]
    repeats  = 3 if sku in TOP30 else 1

    for _ in range(repeats):
        for d in pd.date_range('2024-03-01', '2025-09-04', freq='2D'):
            X_rows.append(build_features(d, series, meta_row))
            y_rows.append(float(series.get(d, 0.0)))

    if (i + 1) % 10 == 0:
        print(f'  {i+1}/{len(TIER_ML)} SKUs — '
              f'{len(X_rows):,} rows so far')

X = pd.DataFrame(X_rows).fillna(0)
y = np.log1p(y_rows)

print(f'\nDone!')
print(f'Training samples : {len(X):,}')
print(f'Features         : {X.shape[1]}')
print(f'Target % zeros   : {(np.array(y_rows)==0).mean():.1%}')

Building training data for 19 ML SKUs...
  10/19 SKUs — 8,310 rows so far

Done!
Training samples : 15,789
Features         : 78
Target % zeros   : 96.0%


In [6]:
# Revenue-based sample weights
# Higher revenue SKU → higher weight → model focuses on it more
rev_max = sku_meta.loc[TIER_ML, 'total_revenue'].max()

sample_weights = []
row_idx = 0
for sku in TIER_ML:
    repeats    = 3 if sku in TOP30 else 1
    n_dates    = len(pd.date_range('2024-03-01', '2025-09-04', freq='2D'))
    n_rows     = n_dates * repeats
    rev_norm   = float(sku_meta.loc[sku, 'total_revenue']) / (rev_max + 1e-8)
    # Weight: 1.0 to 5.0 based on revenue
    w          = 1.0 + 4.0 * rev_norm
    sample_weights.extend([w] * n_rows)

sample_weights = np.array(sample_weights)

print(f'Sample weight stats:')
print(f'  Min  : {sample_weights.min():.3f}')
print(f'  Max  : {sample_weights.max():.3f}')
print(f'  Mean : {sample_weights.mean():.3f}')

Sample weight stats:
  Min  : 1.172
  Max  : 5.000
  Mean : 1.635


In [7]:
print('Training HistGradientBoostingRegressor...')

model = HistGradientBoostingRegressor(
    max_iter          = 500,
    learning_rate     = 0.04,
    max_leaf_nodes    = 47,
    min_samples_leaf  = 15,
    l2_regularization = 0.5,
    random_state      = 42
)
model.fit(X, y, sample_weight=sample_weights)

pred_is = np.maximum(np.expm1(model.predict(X)), 0)
mae     = np.mean(np.abs(pred_is - np.array(y_rows)))
print(f'Training done!')
print(f'In-sample MAE : {mae:.3f}')

Training HistGradientBoostingRegressor...
Training done!
In-sample MAE : 1.890


In [8]:
os.makedirs('../models', exist_ok=True)
FEAT_COLS = X.columns.tolist()

joblib.dump(model,     '../models/hgbr_model.pkl')
joblib.dump(FEAT_COLS, '../models/feat_cols.pkl')
joblib.dump(TOP30,     '../models/top30_skus.pkl')

print(f'Model saved    → ../models/hgbr_model.pkl')
print(f'Features saved → ../models/feat_cols.pkl')
print(f'Top30 saved    → ../models/top30_skus.pkl')
print(f'Features       : {len(FEAT_COLS)}')

Model saved    → ../models/hgbr_model.pkl
Features saved → ../models/feat_cols.pkl
Top30 saved    → ../models/top30_skus.pkl
Features       : 78
